# P-Tuning v2 交互教学

配套 lecture：[`../lectures/04-p-tuning-v2.md`](../lectures/04-p-tuning-v2.md)  
配套论文：[`../papers/04-p-tuning-v2-2022.pdf`](../papers/04-p-tuning-v2-2022.pdf)

本 notebook 演示：
1. 环境检查
2. 手写 minimal 版（无 reparam）
3. peft 调包版
4. **强一致性验证**（logits bit 精确）
5. 与 Prefix Tuning 的代码对比
6. 思考题

In [ ]:
import sys
import torch, transformers, peft
print(f'Python: {sys.version.split()[0]}')
print(f'torch:        {torch.__version__}')
print(f'transformers: {transformers.__version__}')
print(f'peft:         {peft.__version__}')

## 2. 手写 minimal 版

公式 (1)(2)：prefix 形状 (L, 2, p, H, d_h)，**直接学**，无 MLP。

In [ ]:
from pathlib import Path
src_dir = (Path.cwd().parent / 'src').resolve()
sys.path.insert(0, str(src_dir))

from p_tuning_v2_minimal import PTuningV2GPT2
from common import print_param_summary

torch.manual_seed(42)
model = PTuningV2GPT2(prefix_length=10)
print_param_summary(model, 'v2 minimal (p=10, no reparam)')
print(f'\nprefix shape: {tuple(model.prefix.shape)}')
print(f'核算 L*2*p*H*d_h = 12*2*10*12*64 = {12*2*10*12*64:,}')

## 3. peft 调包版（PrefixTuningConfig + prefix_projection=False）

In [ ]:
from p_tuning_v2_peft import build_peft_model
torch.manual_seed(42)
peft_model = build_peft_model(prefix_length=10)
print_param_summary(peft_model, 'v2 peft')
for name, p in peft_model.named_parameters():
    if p.requires_grad:
        print(f'  {name}: {tuple(p.shape)} = {p.numel():,}')
print(f'\npeft 把 prefix 存为 (p, L*2*d) = (10, 18432) 的二维矩阵；')
print(f'minimal 存为 (L, 2, p, H, d_h) 五维张量。')
print(f'两者参数数完全相同 (184,320)，仅布局不同。')

## 4. **强一致性验证**

由于 v2 无 reparam、无 LSTM、无 dropout，两边应数值精确一致。

In [ ]:
tests_dir = (Path.cwd().parent / 'src' / 'tests').resolve()
sys.path.insert(0, str(tests_dir))
from test_p_tuning_v2_consistency import test_logits_match
test_logits_match()

## 5. 与 Prefix Tuning 的对比

v2 ≈ Prefix Tuning - MLP。验证：在相同 prefix_length 下，参数量差异完全来自 MLP。

In [ ]:
from prefix_tuning_minimal import PrefixTuningGPT2
from common import count_parameters

v2 = PTuningV2GPT2(prefix_length=10)
prefix_full = PrefixTuningGPT2(prefix_length=10, mid_dim=512, use_reparam=True)
prefix_no_reparam = PrefixTuningGPT2(prefix_length=10, use_reparam=False)

n_v2, _ = count_parameters(v2)
n_prefix_full, _ = count_parameters(prefix_full)
n_prefix_no, _ = count_parameters(prefix_no_reparam)

print(f'P-Tuning v2 (无 reparam):              {n_v2:>10,}')
print(f'Prefix Tuning (no reparam) ≈ v2:      {n_prefix_no:>10,}')
print(f'Prefix Tuning (含 MLP reparam):       {n_prefix_full:>10,}')
print(f'\n→ v2 与 Prefix Tuning 去 reparam 后参数量{("完全相同" if n_v2 == n_prefix_no else "不同")}')
print(f'→ MLP 多带 {n_prefix_full - n_v2:,} 参数 ≈ {(n_prefix_full - n_v2) / n_v2:.1f}x of v2')

## 6. 思考题与全专题总结

1. **布局转换**：写一个函数 `convert(minimal_prefix)`，把 (L, 2, p, H, d_h) 转成 (p, L*2*d)。
2. **学习率敏感**：v2 论文用 5e-3 ～ 1e-2，Prefix Tuning 用 5e-5。可以理解为什么差 100 倍吗？
3. **deep vs shallow**：把 v2 退化为"只前 4 层加 prefix"，参数量降为多少？
4. **v2 vs Prefix（代码 diff）**：vimdiff `prefix_tuning_minimal.py` 与 `p_tuning_v2_minimal.py`，列出代码层面的全部差异。
5. **全专题串联**：四种方法的核心差异，请用一张表回顾。


In [ ]:
# 全专题参数量串联
from prompt_tuning_minimal import PromptTuningGPT2
from p_tuning_minimal import PTuningGPT2

data = [
    ('Prompt Tuning',  count_parameters(PromptTuningGPT2(prompt_length=10))[0]),
    ('Prefix Tuning',  count_parameters(PrefixTuningGPT2(prefix_length=10, mid_dim=512))[0]),
    ('P-Tuning v1',    count_parameters(PTuningGPT2(prompt_length=10, encoder_hidden=256))[0]),
    ('P-Tuning v2',    count_parameters(PTuningV2GPT2(prefix_length=10))[0]),
]

print(f'{"方法":<18}{"可训练参数":>14}{"占 GPT-2 base ":>18}')
print('-' * 55)
for name, n in data:
    pct = 100 * n / 124_447_488
    print(f'{name:<18}{n:>14,}{pct:>15.4f}%')